In [ ]:
from pathlib import Path
import os, sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/softplus_controller.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


In [ ]:
from src.environments.Environment import *
from src.agents.DDPG import *
from src.models.NN_Module import *
from configs.config_56bus import Config
import os

import torch
import matplotlib.pyplot as plt
import scienceplots
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
from scipy import interpolate

from loguru import logger
import os
import numpy as np

# 方案1 - 柔和专业的配色
# colors = ['#4477AA', '#66CCEE', '#228833', '#CCBB44', '#EE6677', '#AA3377']

# 方案2 - 高对比度但不刺眼的配色
colors = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377']

NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_legend": 24,
    "dpi": 300
}

### a simple logger
logger.remove()
logger.add(sys.stderr, level='INFO')

In [ ]:
env_seed = 10        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l 6l

agent_policy_net = []
safe_agent_net = []

### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
# injection_bus = np.array([15, 18, 19, 21, 23, 30, 34, 45, 46, 53])-1
# pp_net = create_56bus_10()
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
state, topology, senario = env.reset_topo(seed=env_seed,manual_switch={2:False})      #change topology
# state, topology, senario = env.reset(seed=env_seed)             #not change
topology = torch.cuda.FloatTensor(topology).unsqueeze(0)

agent_num = env.agentnum

simple_plotly(env.network, figsize=1)
pf_res_plotly(pp_net);

In [ ]:
### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=Config.topology_hidden_dim)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=Config.hidden_dim_56bus).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'))
    # policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-09-17/Step_200_Seed_45_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(5):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg/policy_net_checkpoint_a{i}.pth')

    safe_agent_net[i].load_state_dict(policy_net_dict)


In [ ]:
def run_with_disabled_agents(env, agent_policy_net, agent_num, disabled_agents=None, env_seed=1):
    """
    Run the environment with the ability to disable specific agents.

    Parameters:
    - env: The environment instance.
    - agent_policy_net: List of agent policy networks.
    - agent_num: Total number of agents.
    - disabled_agents: List of integers, indices of agents to disable (set their action to 0).
    - env_seed: Seed for environment reset.

    Returns:
    - voltage_RL: Numpy array of voltage trajectories.
    - voltage_all_RL: Numpy array of all voltage states.
    - q_RL: Numpy array of actions.
    """
    if disabled_agents is None:
        disabled_agents = []

    episode_reward = 0
    episode_control = 0
    voltage = []
    voltage_all = []
    q = []
    cost = []

    last_action = np.zeros((agent_num, 1))

    done_record = True
    state, topology, senario = env.reset_topo(seed=env_seed, manual_switch={2: False})  # change topology
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)

    for t in range(300):
        action = []
        for i in range(agent_num):
            if i in disabled_agents:
                action_agent = 0  # Disable this agent's action
            else:
                action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
                action_agent = action_agent.detach().cpu().numpy()
                action_agent = action_agent.item()  # Ensure action_agent is a scalar
            action.append(action_agent)

        if np.min(action) < -1.0 or np.max(action) > 1.0:
            logger.warning('control output saturated! min is {}, max is {}', np.min(action), np.max(action))

        action = last_action - np.asarray(action).reshape(-1, 1)  # Ensure action is a 2D array
        last_action = np.copy(action)

        try:
            next_state, reward, done = env.step(action)
        except:
            logger.error(sys.exc_info())
            logger.error('power flow not converge at {}', t)
            break

        if done and done_record:
            logger.info('RLC-FT stable at step {}', t)
            logger.info('RLC-FT stable cost is {}', episode_control)
            done_record = False

        voltage.append(state)
        voltage_all.append(env.get_all_state())
        q.append(action)

        state = next_state
        episode_reward += reward
        cost.append(-reward)
        episode_control += LA.norm(action, 2)

    voltage_RL = np.asarray(voltage)
    voltage_all_RL = np.asarray(voltage_all)
    q_RL = np.asarray(q)

    return voltage_RL, voltage_all_RL, q_RL

# Example usage:
disabled_agents = [0,1,3,4]  
voltage_RL, voltage_all_RL, q_RL = run_with_disabled_agents(env, agent_policy_net, agent_num, disabled_agents=disabled_agents, env_seed=10)

In [ ]:
print(pp_net.sgen)
print(voltage_all_RL.shape)
print(voltage_RL.shape)

In [ ]:
def plot_voltage_trajectory_mean(
    voltage_RL,
    scenario=1,
    topology_change_steps=None,
    bus_indices=(0, 1, 2, 3, 4),
    max_steps=30,                 # 只保留前 N 个 step（含 0）
    show_markers=True,
    marker_every=1,               # 每隔几个点加一个 marker
    show_range=False,             # 是否显示 min--max 阴影带
    event_style='line',           # 'line' 用虚线, 'band' 用灰带
    event_labels=None,            # 自定义事件标签列表；默认 ['Disconnection','Reconnection',...]
    export_path_svg=None,
    export_path_png=None
):
    """
    Plot mean bus voltage (kV) over selected buses with topology-change markers.
    voltage_RL: (T, B) per-unit, converted to kV by *12.0
    """
    import numpy as np
    import plotly.graph_objects as go
    import plotly.io as pio
    import os

    T, B = voltage_RL.shape
    last_idx = min(int(max_steps), T-1) if max_steps is not None else T-1
    steps = np.arange(last_idx + 1)

    # Data in kV
    y_sel  = 12.0 * voltage_RL[:last_idx+1, list(bus_indices)]   # (N, K)
    y_mean = np.mean(y_sel, axis=1)
    y_min  = np.min(y_sel, axis=1)
    y_max  = np.max(y_sel, axis=1)

    # Colors
    mean_color    = '#0072B2'                         # blue
    range_fill    = 'rgba(0,114,178,0.18)'            # light blue alpha
    limit_color   = '#009E73'                         # green
    nominal_color = '#666666'                         # gray
    event_line_color = 'rgba(0,0,0,0.55)'             # subtle gray for event lines

    # Axis ranges by scenario
    if scenario == 0:
        y_low, y_high = 10.0, 12.6
        upper_limit, lower_limit = None, 11.4
    elif scenario == 1:
        y_low, y_high = 11.8, 13.5
        upper_limit, lower_limit = 12.6, None
    else:
        y_low, y_high = 10.8, 13.2
        upper_limit, lower_limit = 12.6, 11.4

    # Padding based on shown segment
    y_low  = min(y_low,  float(np.min(y_min)) - 0.05)
    y_high = max(y_high, float(np.max(y_max)) + 0.05)

    fig = go.Figure()

    # Event markers (line or band)
    if topology_change_steps is not None:
        if not isinstance(topology_change_steps, (list, tuple, np.ndarray)):
            topology_change_steps = [int(topology_change_steps)]
        if event_labels is None:
            # 默认给前两个事件以更具体的名称
            default_names = ['Disconnection', 'Reconnection']
            event_labels = [default_names[i] if i < len(default_names) else f'Change {i+1}'
                            for i in range(len(topology_change_steps))]
        if event_style == 'band':
            for step, lab in zip(topology_change_steps, event_labels):
                if step < 0 or step > steps[-1]: 
                    continue
                x0 = max(step - 0.5, steps[0])
                x1 = min(step + 0.5, steps[-1])
                fig.add_vrect(x0=x0, x1=x1, fillcolor='rgba(0,0,0,0.18)', line_width=0)
                fig.add_annotation(x=step, y=y_high, xref='x', yref='y',
                                   text=lab, showarrow=False, yshift=10,
                                   font=dict(size=NATURE_CONFIG.get("font_axis", 24), color='#333'))
        else:  # 'line' style (recommended with range band)
            for step, lab in zip(topology_change_steps, event_labels):
                if step < 0 or step > steps[-1]:
                    continue
                fig.add_vline(x=step, line=dict(color=event_line_color, width=1.6, dash='dash'))
                # place label above the plotting area to avoid overlap with bands
                fig.add_annotation(x=step, xref='x', y=1.08, yref='paper',
                                   text=lab, showarrow=False,
                                   font=dict(size=NATURE_CONFIG.get("font_axis",24), color='#444'),
                                   align='center')

    # Optional min--max envelope
    if show_range:
        fig.add_trace(go.Scatter(
            x=steps, y=y_max,
            line=dict(color='rgba(0,0,0,0)'), showlegend=False, hoverinfo='skip'
        ))
        fig.add_trace(go.Scatter(
            x=steps, y=y_min,
            fill='tonexty', fillcolor=range_fill,
            line=dict(color='rgba(0,0,0,0)'), showlegend=False, hoverinfo='skip'
        ))

    # Mean trajectory (with optional markers)
    if show_markers:
        line_trace = go.Scatter(
            x=steps, y=y_mean, mode='lines',
            line=dict(color=mean_color, width=2.4),
            hoverinfo='skip', showlegend=False
        )
        marker_trace = go.Scatter(
            x=steps[::max(1, marker_every)],
            y=y_mean[::max(1, marker_every)],
            mode='markers',
            marker=dict(size=10, color=mean_color, line=dict(color='white', width=0.5)),
            name=f'Mean voltage (n={len(bus_indices)})',
            hovertemplate='Step=%{x}<br>Mean V=%{y:.3f} kV<extra></extra>'
        )
        fig.add_trace(line_trace)
        fig.add_trace(marker_trace)
    else:
        fig.add_trace(go.Scatter(
            x=steps, y=y_mean, mode='lines',
            name=f'Mean voltage (n={len(bus_indices)})',
            line=dict(color=mean_color, width=2.4),
            hovertemplate='Step=%{x}<br>Mean V=%{y:.3f} kV<extra></extra>'
        ))

    # Limits and labels (offset to avoid overlap)
    if upper_limit is not None:
        fig.add_hline(y=upper_limit, line=dict(color=limit_color, width=2))
        fig.add_annotation(
            x=0.985, xref='paper', y=upper_limit, yref='y',
            text='Voltage Upper Limit',
            showarrow=False, xanchor='left',
            yshift=15, xshift=-150,
            font=dict(size=20, color=limit_color),
            bgcolor='rgba(255,255,255,0.7)'
        )
    if lower_limit is not None:
        fig.add_hline(y=lower_limit, line=dict(color=limit_color, width=2, dash='dash'))

    fig.add_hline(y=12.0, line=dict(color=nominal_color, width=1.5))
    fig.add_annotation(
        x=0.985, xref='paper', y=12.0, yref='y',
        text='Nominal Voltage',
        showarrow=False, xanchor='left',
        yshift=-15, xshift=-150,
        font=dict(size=20, color=nominal_color),
        bgcolor='rgba(255,255,255,0.7)'
    )

    # Layout
    width  = 700
    height = 500
    fig.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG.get("font_base",28)-4, color='black'),
        width=width, height=height,
        margin=dict(l=60, r=60, t=60, b=60),  # Adjust margins to give more space
        plot_bgcolor='white', paper_bgcolor='white',
        showlegend=False,
        xaxis=dict(
            range=[steps[0] - 0.5, steps[-1] + 0.5],  # Add padding to avoid marker clipping
            title='Iteration Steps',
            showgrid=True, gridcolor='#E6E6E6', gridwidth=0.5,
            zeroline=False, 
            dtick=10,
            title_standoff=20,  # Add space between axis title and numbers
            tickfont=dict(size=NATURE_CONFIG.get("font_axis", 24) - 2),
        ),
        yaxis=dict(
            range=[y_low - 0.1, y_high + 0.1],  # Add padding to avoid marker clipping
            title='Bus voltage (kV)',
            showgrid=True, gridcolor='#E6E6E6', gridwidth=0.5,
            zeroline=False, 
            dtick=0.4 if (y_high - y_low) <= 3.0 else 0.5,
            title_standoff=20,  # Add space between axis title and numbers
            tickfont=dict(size=NATURE_CONFIG.get("font_axis", 24) - 2),
        ),
    )

    # Export images
    # if export_path_svg is None:
    #     export_path_svg = os.path.join(Config.data_path, 'images', '56bus', 'voltage_plot_subset.svg')
    # if export_path_png is None:
    #     export_path_png = os.path.join(Config.data_path, 'images', '56bus', 'voltage_plot_subset.png')
    # fig.write_image(export_path_svg, scale=2)
    # fig.write_image(export_path_png, scale=2)
    fig.show()
    return fig

# Example usage:
fig = plot_voltage_trajectory_mean(voltage_RL, scenario=1,
                                   show_range=True, max_steps=100, event_style='line')

In [ ]:
def plot_voltage_trajectory_all_buses(
    voltage_RL,
    bus_indices=(0, 1, 2, 3, 4),
    max_steps=30,                 # 只保留前 N 个 step（含 0）
    show_markers=True,
    marker_every=1,               # 每隔几个点加一个 marker
    export_path_svg=None,
    export_path_png=None
):
    """
    Plot bus voltage (kV) trajectories for all selected buses.
    voltage_RL: (T, B) per-unit, converted to kV by *12.0
    """
    import numpy as np
    import plotly.graph_objects as go

    T, B = voltage_RL.shape
    last_idx = min(int(max_steps), T-1) if max_steps is not None else T-1
    steps = np.arange(last_idx + 1)

    # Data in kV
    y_sel = 12.0 * voltage_RL[:last_idx+1, list(bus_indices)]   # (N, K)

    # Colors
    bus_colors = ['#0072B2', '#009E73', '#D55E00', '#CC79A7', '#F0E442', '#56B4E9', '#E69F00', '#999999']
    nominal_color = '#666666'  # gray
    limit_color = '#009E73'    # green

    fig = go.Figure()

    # Plot each bus trajectory
    for i, bus_idx in enumerate(bus_indices):
        color = bus_colors[i % len(bus_colors)]
        if show_markers:
            line_trace = go.Scatter(
                x=steps, y=y_sel[:, i], mode='lines',
                line=dict(color=color, width=2.4),
                hoverinfo='skip', showlegend=True, name=f'Bus {bus_idx + 1}'
            )
            marker_trace = go.Scatter(
                x=steps[::max(1, marker_every)],
                y=y_sel[::max(1, marker_every), i],
                mode='markers',
                marker=dict(size=8, color=color, line=dict(color='white', width=0.5)),
                hovertemplate=f'Bus {bus_idx + 1}<br>Step=%{{x}}<br>Voltage=%{{y:.3f}} kV<extra></extra>',
                showlegend=False
            )
            fig.add_trace(line_trace)
            fig.add_trace(marker_trace)
        else:
            fig.add_trace(go.Scatter(
                x=steps, y=y_sel[:, i], mode='lines',
                name=f'Bus {bus_idx + 1}',
                line=dict(color=color, width=2.4),
                hovertemplate=f'Bus {bus_idx + 1}<br>Step=%{{x}}<br>Voltage=%{{y:.3f}} kV<extra></extra>'
            ))

    # Add upper and lower limits
    fig.add_hline(y=12.6, line=dict(color=limit_color, width=2))
    fig.add_annotation(
        x=0.985, xref='paper', y=12.6, yref='y',
        text='Upper Limit (12.6 kV)',
        showarrow=False, xanchor='left',
        yshift=15, xshift=-150,
        font=dict(size=20, color=limit_color),
        bgcolor='rgba(255,255,255,0.7)'
    )
    fig.add_hline(y=11.4, line=dict(color=limit_color, width=2, dash='dash'))
    fig.add_annotation(
        x=0.985, xref='paper', y=11.4, yref='y',
        text='Lower Limit (11.4 kV)',
        showarrow=False, xanchor='left',
        yshift=-15, xshift=-150,
        font=dict(size=20, color=limit_color),
        bgcolor='rgba(255,255,255,0.7)'
    )

    # Nominal voltage line
    fig.add_hline(y=12.0, line=dict(color=nominal_color, width=1.5))
    fig.add_annotation(
        x=0.985, xref='paper', y=12.0, yref='y',
        text='Nominal Voltage',
        showarrow=False, xanchor='left',
        yshift=-15, xshift=-150,
        font=dict(size=20, color=nominal_color),
        bgcolor='rgba(255,255,255,0.7)'
    )

    # Layout
    width = 700
    height = 500
    fig.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG.get("font_base", 28) - 4, color='black'),
        width=width, height=height,
        margin=dict(l=60, r=60, t=60, b=60),  # Adjust margins to give more space
        plot_bgcolor='white', paper_bgcolor='white',
        showlegend=True,
        xaxis=dict(
            range=[steps[0] - 0.5, steps[-1] + 0.5],  # Add padding to avoid marker clipping
            title='Iteration Steps',
            showgrid=True, gridcolor='#E6E6E6', gridwidth=0.5,
            zeroline=False,
            dtick=10,
            title_standoff=20,  # Add space between axis title and numbers
            tickfont=dict(size=NATURE_CONFIG.get("font_axis", 24) - 2),
        ),
        yaxis=dict(
            range=[11.0, 13.0],  # Fixed range for better comparison
            title='Bus voltage (kV)',
            showgrid=True, gridcolor='#E6E6E6', gridwidth=0.5,
            zeroline=False,
            dtick=0.2,
            title_standoff=20,  # Add space between axis title and numbers
            tickfont=dict(size=NATURE_CONFIG.get("font_axis", 24) - 2),
        ),
    )

    # Export images
    if export_path_svg:
        fig.write_image(export_path_svg, scale=2)
    if export_path_png:
        fig.write_image(export_path_png, scale=2)
    fig.show()
    return fig

# Example usage:
bus_indices = tuple(range(voltage_all_RL.shape[1]))  # Dynamically determine bus indices based on voltage_RL
# bus_indices = (12,13,36,37,42,43,49)
fig = plot_voltage_trajectory_all_buses(voltage_all_RL, show_markers=True, marker_every=2, bus_indices=bus_indices, max_steps=50)

In [ ]:

def plot_voltage_trajectory_all_buses(
    voltage_RL,
    bus_indices=(0, 1, 2, 3, 4),
    max_steps=30,
    show_markers=True,
    marker_every=1,
    export_path_svg=None,
    export_path_png=None
):
    """
    Plot bus voltage (kV) trajectories.
    Selected buses are highlighted with color and markers.
    Other buses are shown as a gray background.
    
    voltage_RL: (T, B) per-unit, converted to kV by *12.0
    bus_indices: A tuple of integer indices for buses to highlight.
    """
    import numpy as np
    import plotly.graph_objects as go

    T, total_buses = voltage_RL.shape
    last_idx = min(int(max_steps), T-1) if max_steps is not None else T-1
    steps = np.arange(last_idx + 1)

    # Data in kV for all buses
    y_data = 12.0 * voltage_RL[:last_idx+1, :]  # (N, B)

    # --- NEW: Identify selected vs. unselected buses ---
    all_bus_indices = set(range(total_buses))
    selected_bus_indices = set(bus_indices)
    unselected_bus_indices = sorted(list(all_bus_indices - selected_bus_indices))

    # --- Colors ---
    bus_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
]
#     bus_colors = [
#     '#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE',
#     '#AA3377', '#BBBBBB' 
# ] 
    nominal_color = '#666666'
    limit_color = '#009E73'
    background_color = '#D3D3D3'  # Light gray for unselected buses

    fig = go.Figure()

    # --- NEW: First, plot all unselected buses as a gray background ---
    for bus_idx in unselected_bus_indices:
        fig.add_trace(go.Scatter(
            x=steps,
            y=y_data[:, bus_idx],
            mode='markers+lines',
            line=dict(color=background_color, width=2),
            hoverinfo='none',  # Hide hover info for background lines
            showlegend=False   # Do not show in legend
        ))

    # --- MODIFIED: Second, plot the selected buses on top ---
    for i, bus_idx in enumerate(bus_indices):
        color = bus_colors[i % len(bus_colors)]
        
        # We need the specific column for this bus from the full y_data
        y_sel_bus = y_data[:, bus_idx]
        
        if show_markers:
            line_trace = go.Scatter(
                x=steps, y=y_sel_bus, mode='lines',
                line=dict(color=color, width=2.4),
                hoverinfo='skip',
                showlegend=True, name=f'Bus {bus_idx}' # This will appear in the legend
            )
            marker_trace = go.Scatter(
                x=steps[::max(1, marker_every)],
                y=y_sel_bus[::max(1, marker_every)],
                mode='markers',
                marker=dict(size=8, color=color, line=dict(color='white', width=0.5)),
                hovertemplate=f'Bus {bus_idx}<br>Step=%{{x}}<br>Voltage=%{{y:.3f}} kV<extra></extra>',
                showlegend=False # Markers don't need a separate legend entry
            )
            fig.add_trace(line_trace)
            fig.add_trace(marker_trace)
        else:
            fig.add_trace(go.Scatter(
                x=steps, y=y_sel_bus, mode='lines',
                name=f'Bus {bus_idx}',
                line=dict(color=color, width=2.4),
                hovertemplate=f'Bus {bus_idx}<br>Step=%{{x}}<br>Voltage=%{{y:.3f}} kV<extra></extra>'
            ))

    # --- The rest of the code remains unchanged ---

    # Add upper and lower limits
    fig.add_hline(y=12.6, line=dict(color=limit_color, width=2))
    fig.add_annotation(
        x=0.985, xref='paper', y=12.6, yref='y',
        text='Upper Limit (12.6 kV)',
        showarrow=False, xanchor='left',
        yshift=15, xshift=-200,
        font=dict(size=20, color=limit_color),
        bgcolor='rgba(255,255,255,0.7)'
    )
    fig.add_hline(y=11.4, line=dict(color=limit_color, width=2, dash='dash'))
    fig.add_annotation(
        x=0.985, xref='paper', y=11.4, yref='y',
        text='Lower Limit (11.4 kV)',
        showarrow=False, xanchor='left',
        yshift=-15, xshift=-200,
        font=dict(size=20, color=limit_color),
        bgcolor='rgba(255,255,255,0.7)'
    )

    # Nominal voltage line
    fig.add_hline(y=12.0, line=dict(color=nominal_color, width=1.5))
    # fig.add_annotation(
    #     x=0.985, xref='paper', y=12.0, yref='y',
    # text='Nominal Voltage',
    #     showarrow=False, xanchor='left',
    #     yshift=-15, xshift=-150,
    #     font=dict(size=20, color=nominal_color),
    #     bgcolor='rgba(255,255,255,0.7)'
    # )

    # Layout
    width = 700
    height = 500
    fig.update_layout(
        font=dict(family='Arial', size=NATURE_CONFIG.get("font_base", 28) - 4, color='black'),
        width=width, height=height,
        margin=dict(l=60, r=60, t=60, b=60),
        plot_bgcolor='white', paper_bgcolor='white',
        showlegend=True, # Ensure legend is visible
        xaxis=dict(
            range=[steps[0] - 0.5, steps[-1] + 0.5],
            title='Iteration Steps',
            showgrid=True, gridcolor='#E6E6E6', gridwidth=0.5,
            zeroline=False,
            dtick=10,
            title_standoff=20,
            tickfont=dict(size=NATURE_CONFIG.get("font_axis", 24) - 2),
        ),
        yaxis=dict(
            range=[11.0, 13.0],
            title='Bus voltage (kV)',
            showgrid=True, gridcolor='#E6E6E6', gridwidth=0.5,
            zeroline=False,
            dtick=0.2,
            title_standoff=20,
            tickfont=dict(size=NATURE_CONFIG.get("font_axis", 24) - 2),
        ),
    )

    # Export images
    if export_path_svg:
        fig.write_image(export_path_svg, scale=2)
    if export_path_png:
        fig.write_image(export_path_png, scale=2)
    fig.show()
    return fig

# Select a few buses to highlight
highlighted_buses = (30,)

fig = plot_voltage_trajectory_all_buses(
    voltage_all_RL, 
    bus_indices=highlighted_buses, 
    show_markers=True, 
    marker_every=2, 
    max_steps=50,
    export_path_svg=os.path.join(Config.data_path, 'images', '56bus', 'voltage_plot_1_buses.svg'),
    export_path_png=os.path.join(Config.data_path, 'images', '56bus', 'voltage_plot_1_buses.png')
)

In [ ]:
### test our controller

### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=Config.topology_hidden_dim)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=Config.hidden_dim_56bus).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    # policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'))
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-09-17/Step_200_Seed_45_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)


voltage = []
q = []
cost = []
success_list = []
fail_list = []
entire_list = []
control_cost = []
reward_list = []
object_cost = []
voltage_trajs = []

for episode in range(200):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost = []
    abnormal_stop = False

    voltage_episode = []   # stores full voltage vectors for this episode

    for step in range(200):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = action_agent.detach().cpu().numpy()[0] 
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list[-1][0], success_list[-1][1])
            break

        voltage.append(state)
        voltage_episode.append(state.copy())  # record full state vector

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list.append(episode_reward)
    control_cost.append(episode_control)
    object_cost.append(np.sum(cost))
    if len(voltage_episode) > 0 and senario == 0:
        voltage_trajs.append(np.vstack(voltage_episode))

    if (not done) and (abnormal_stop == False):
        entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list))
logger.info('total fail episode is {}', len(fail_list))
logger.info('number of finished at entire episode is {}', len(entire_list))